##DBTC-1203 
### Delete of duplicate records

In [0]:
spark.sql(f"""
DELETE
FROM `p-cdh`.`landed`.process_settlement_loan
WHERE participation_perc IS NULL
AND BusinessDate = '2025-01-29'
""")

##D360-2229: Addition of Segment and Strategy in dbpcepalrecoveriesparty

In [0]:
env="p"

In [0]:
spark.sql(f'''alter table `{env}-cdh`.derivativeactive.dbpcepalrecoveriesparty
add column Segment string after BaseDate;''')

In [0]:
spark.sql(f'''alter table `{env}-cdh`.derivativeactive.dbpcepalrecoveriesparty
add column Strategy string after Segment;''')

##DATA2-236 - Portfolio update

In [0]:
%sql
UPDATE `p-cdh`.primaryactive.portfolio
SET AssetOwner='Monza NPL Finance DAC'
where portfolioname like '%MONZA%'

In [0]:
%sql
UPDATE `p-cdh`.primaryactive.portfolio
SET Short_Name_Portfolio_Name='Waterwheel'
where portfolioname like '%MONZA%' ;
 

##Refresh views

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)



In [0]:
# Get the schema of the dexposure table
dexposure_table = spark.table(f"`{env}-cdh`.derivative.dexposure")
dexposure_schema_fields = dexposure_table.schema.fields


# Convert the schema fields into the required string format
dexposure_schema = ",\n".join([
    f"{field.name} {field.dataType.simpleString()}"
    for field in dexposure_schema_fields
])


spark.sql(f'''
CREATE OR REPLACE FUNCTION `{env}-cdh`.derivative.get_all_exposures_by_business_date(
    business_date Date
)
RETURNS TABLE 
( 
{dexposure_schema}
)
RETURN

SELECT B.*
FROM(
        SELECT  ExposureKey, MAX(BusinessDate) BusinessDate
        FROM `{env}-cdh`.derivative.dexposure
        WHERE BusinessDate <= business_date
        GROUP BY ExposureKey
) A
INNER JOIN   `{env}-cdh`.derivative.dexposure B
on a.ExposureKey = b.ExposureKey
and a.BusinessDate = b.BusinessDate
AND B.IsActiveFlagInt = 0

UNION

SELECT * FROM `{env}-cdh`.derivative.dexposure
WHERE BusinessDate = business_date
AND IsActiveFlagInt = 1
''')